<a href="https://colab.research.google.com/github/DevKnight-code/Supply-Chain-Inventory-Optimization/blob/main/python/Supply_Chain_Inventory_Optimization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

print("Libraries loaded successfully.")

Libraries loaded successfully.


In [ ]:
from google.colab import files

excel_upload = files.upload()

excel_file = list(excel_upload.keys())[0]

print("Excel file:", excel_file)

Saving SupplyChain_Inventory_Planning_Analysis.xlsm to SupplyChain_Inventory_Planning_Analysis.xlsm
Excel file: SupplyChain_Inventory_Planning_Analysis.xlsm


In [ ]:
print("Select Demand_Forecast_Results.csv")

forecast_upload = files.upload()

forecast_file = list(forecast_upload.keys())[0]

print("Forecast file:", forecast_file)

Select Demand_Forecast_Results.csv


Saving Demand_Forecast_Results.csv to Demand_Forecast_Results.csv
Forecast file: Demand_Forecast_Results.csv


In [ ]:
print("Select Stockout_Risk_Results.csv")

stockout_upload = files.upload()

stockout_file = list(stockout_upload.keys())[0]

print("Stockout file:", stockout_file)

Select Stockout_Risk_Results.csv


Saving Stockout_Risk_Results.csv to Stockout_Risk_Results.csv
Stockout file: Stockout_Risk_Results.csv


In [ ]:
import os

print("Excel file:")
print(excel_file)

print("\nForecast file:")
print(forecast_file)

print("\nStockout file:")
print(stockout_file)

print("\nFiles currently available in Colab:")
print(os.listdir("/content"))

Excel file:
SupplyChain_Inventory_Planning_Analysis.xlsm

Forecast file:
Demand_Forecast_Results.csv

Stockout file:
Stockout_Risk_Results.csv

Files currently available in Colab:
['.config', 'Demand_Forecast_Results.csv', '.ipynb_checkpoints', 'SupplyChain_Inventory_Planning_Analysis.xlsm', 'Stockout_Risk_Results.csv', 'sample_data']


In [ ]:
optimization = pd.read_excel(
    excel_file,
    sheet_name="Optimization_Data"
)

forecast = pd.read_csv(
    forecast_file
)

stockout = pd.read_csv(
    stockout_file
)

print("Optimization:", optimization.shape)
print("Forecast    :", forecast.shape)
print("Stockout    :", stockout.shape)

Optimization: (264, 29)
Forecast    : (17160, 12)
Stockout    : (18216, 13)


In [ ]:
forecast = forecast.sort_values(
    [
        "ProductID",
        "WarehouseID",
        "Date"
    ]
)
latest_forecast = (
    forecast
    .groupby(
        ["ProductID", "WarehouseID"],
        as_index=False
    )
    .tail(1)
    .reset_index(drop=True)
)

In [ ]:
print(
    "Original forecast rows:",
    len(forecast)
)

print(
    "Latest forecast rows:",
    len(latest_forecast)
)

display(
    latest_forecast.head()
)

Original forecast rows: 17160
Latest forecast rows: 264


,Date,ProductID,ProductName,Category,WarehouseID,WarehouseName,ActualDemand_7D,ForecastDemand_7D,ActualDemand_14D,ForecastDemand_14D,ActualDemand_30D,ForecastDemand_30D
0,2025-12-01,PROD0001,Dairy Item 1,Grocery,WH01,Northeast DC,359.00,351.27,698.00,691.96,"1,641.00","1,424.34"
1,2025-12-01,PROD0001,Dairy Item 1,Grocery,WH02,Central DC,491.00,372.89,970.00,689.33,"1,815.00","1,393.69"
2,2025-12-01,PROD0002,Camping Gear Item 2,Sports & Outdoors,WH01,Northeast DC,89.00,69.95,183.00,137.76,373.00,292.28
3,2025-12-01,PROD0002,Camping Gear Item 2,Sports & Outdoors,WH02,Central DC,65.00,60.21,135.00,120.20,303.00,256.41
4,2025-12-01,PROD0002,Camping Gear Item 2,Sports & Outdoors,WH03,West Coast DC,102.00,92.52,219.00,181.04,490.00,384.20


In [ ]:
stockout = stockout.sort_values(
    [
        "ProductID",
        "WarehouseID",
        "Date"
    ]
)

latest_stockout = (
    stockout
    .groupby(
        ["ProductID", "WarehouseID"],
        as_index=False
    )
    .tail(1)
    .reset_index(drop=True)
)

In [ ]:
print(
    "Original stockout rows:",
    len(stockout)
)

print(
    "Latest stockout rows:",
    len(latest_stockout)
)

Original stockout rows: 18216
Latest stockout rows: 264


In [ ]:
latest_forecast = latest_forecast[
    [
        "ProductID",
        "WarehouseID",
        "Date",
        "ForecastDemand_7D",
        "ForecastDemand_14D",
        "ForecastDemand_30D"
    ]
].copy()

latest_forecast = latest_forecast.rename(
    columns={
        "Date": "ForecastDate"
    }
)

In [ ]:
latest_stockout = latest_stockout[
    [
        "ProductID",
        "WarehouseID",
        "Date",
        "StockoutProbability",
        "PredictedStockout",
        "StockoutRisk"
    ]
].copy()

latest_stockout = latest_stockout.rename(
    columns={
        "Date": "RiskPredictionDate"
    }
)

In [ ]:
ml_optimization = optimization.merge(
    latest_forecast,
    on=[
        "ProductID",
        "WarehouseID"
    ],
    how="left"
)

In [ ]:
ml_optimization = ml_optimization.merge(
    latest_stockout,
    on=[
        "ProductID",
        "WarehouseID"
    ],
    how="left"
)

In [ ]:
print(
    "Optimization rows:",
    len(optimization)
)

print(
    "Final rows:",
    len(ml_optimization)
)

print(
    "Final columns:",
    len(ml_optimization.columns)
)

Optimization rows: 264
Final rows: 264
Final columns: 37


In [ ]:
print(
    "Missing StockoutProbability:",
    ml_optimization[
        "StockoutProbability"
    ].isna().sum()
)

print(
    "Available StockoutProbability:",
    ml_optimization[
        "StockoutProbability"
    ].notna().sum()
)

print(
    "Total rows:",
    len(ml_optimization)
)

Missing StockoutProbability: 0
Available StockoutProbability: 264
Total rows: 264


In [ ]:
ml_optimization["ForecastDailyDemand_7D"] = (
    ml_optimization["ForecastDemand_7D"] / 7
)

ml_optimization["ForecastDailyDemand_14D"] = (
    ml_optimization["ForecastDemand_14D"] / 14
)

ml_optimization["ForecastDailyDemand_30D"] = (
    ml_optimization["ForecastDemand_30D"] / 30
)

In [ ]:
display(
    ml_optimization[
        [
            "ProductID",
            "WarehouseID",
            "AverageDailyDemand",
            "ForecastDailyDemand_7D",
            "ForecastDailyDemand_14D",
            "ForecastDailyDemand_30D"
        ]
    ].head(20)
)

,ProductID,WarehouseID,AverageDailyDemand,ForecastDailyDemand_7D,ForecastDailyDemand_14D,ForecastDailyDemand_30D
0,PROD0038,WH04,12.74,15.92,16.47,16.53
1,PROD0040,WH04,19.56,16.97,16.82,17.83
2,PROD0031,WH04,10.83,12.70,13.05,13.70
3,PROD0006,WH01,37.38,36.81,37.14,36.76
4,PROD0011,WH01,20.54,25.36,24.81,25.00
5,PROD0057,WH01,3.53,5.33,5.31,5.30
6,PROD0083,WH02,10.93,9.27,9.39,9.37
7,PROD0077,WH01,2.35,2.22,2.23,2.14
8,PROD0003,WH04,22.68,24.82,23.91,22.94
9,PROD0027,WH04,25.16,23.59,23.62,23.17


In [ ]:
ml_optimization["ML_ForecastDailyDemand"] = (
    ml_optimization["ForecastDailyDemand_14D"]
)

In [ ]:
ml_optimization["ML_SafetyStock_Base"] = (
    ml_optimization["ZScore"]
    * ml_optimization["DemandStdDev"]
    * np.sqrt(
        ml_optimization["LeadTime"]
    )
)

In [ ]:
ml_optimization["ML_SafetyStock_Base"] = (
    np.ceil(
        ml_optimization["ML_SafetyStock_Base"]
    )
)

In [ ]:
def risk_multiplier(probability):

    if probability >= 0.60:
        return 1.20

    elif probability >= 0.40:
        return 1.10

    else:
        return 1.00

In [ ]:
ml_optimization["RiskMultiplier"] = (
    ml_optimization[
        "StockoutProbability"
    ].apply(risk_multiplier)
)

In [ ]:
ml_optimization["ML_SafetyStock"] = np.ceil(
    ml_optimization["ML_SafetyStock_Base"]
    * ml_optimization["RiskMultiplier"]
)

In [ ]:
display(
    ml_optimization[
        [
            "ProductID",
            "WarehouseID",
            "StockoutProbability",
            "StockoutRisk",
            "RiskMultiplier",
            "SafetyStock",
            "ML_SafetyStock_Base",
            "ML_SafetyStock"
        ]
    ]
    .sort_values(
        "StockoutProbability",
        ascending=False
    )
    .head(20)
)

In [ ]:
ml_optimization["ML_LeadTimeDemand"] = (
    ml_optimization["ML_ForecastDailyDemand"]
    * ml_optimization["LeadTime"]
)
ml_optimization["ML_LeadTimeDemand"] = np.ceil(
    ml_optimization["ML_LeadTimeDemand"]
)
display(
    ml_optimization[
        [
            "ProductID",
            "WarehouseID",
            "ML_ForecastDailyDemand",
            "LeadTime",
            "ML_LeadTimeDemand"
        ]
    ].head(20)
)

In [ ]:
ml_optimization["ML_ReorderPoint"] = np.ceil(
    ml_optimization["ML_LeadTimeDemand"]
    +
    ml_optimization["ML_SafetyStock"]
)

In [ ]:
display(
    ml_optimization[
        [
            "ProductID",
            "WarehouseID",
            "CurrentStock",
            "ReorderPoint",
            "ML_ReorderPoint",
            "StockoutProbability"
        ]
    ].head(20)
)

In [ ]:
ml_optimization["ML_ReorderFlag"] = np.where(
    ml_optimization["CurrentStock"]
    <= ml_optimization["ML_ReorderPoint"],
    1,
    0
)

In [ ]:
print(
    ml_optimization[
        "ML_ReorderFlag"
    ].value_counts(
        normalize=True
    ) * 100
)

ML_ReorderFlag
1   50.76
0   49.24
Name: proportion, dtype: float64


In [ ]:
ml_optimization.to_csv(
    "ML_Optimization_Base.csv",
    index=False
)

print("ML_Optimization_Base.csv saved successfully.")
print("Rows:", len(ml_optimization))
print("Columns:", len(ml_optimization.columns))

ML_Optimization_Base.csv saved successfully.
Rows: 264
Columns: 47


In [ ]:
from google.colab import files

files.download(
    "ML_Optimization_Base.csv"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>